# npe model family comparison

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# NPE model-family comparison: kinematics only vs mass-ratio conditioned

This notebook compares the existing affine MAF family with a neural spline flow (NSF) and a Gaussian mixture-density network (MDN). We use only the FLAMINGO different-host population because the Local Group is diff-host and holds the row population, connected-host train/test split, features, prior, optimizer settings, and observational scenarios fixed.

The two feature sets are `separation + vr + vt` and the same features plus `log_mass_ratio`. Outputs are written under `pickles_v3/` without modifying the fixed primary checkpoints. Re-running the notebook resumes after the last completed fit.

The comparison is a robustness study.


In [ ]:
# Papermill parameters.
SEEDS_CSV = "42,43,44"
N_POSTERIOR_DRAWS = 200
N_LG_SAMPLES = 10_000
N_OBS_DRAWS = 1_000
N_OBS_SAMPLES_PER_DRAW = 10
CALIBRATION_BATCH_SIZE = 2_048
MAX_NUM_EPOCHS = 200
STOP_AFTER_EPOCHS = 20
EXPECTED_FLAMINGO_FULL = 142_713
EXPECTED_FLAMINGO_DIFF_HOST = 79_642

In [ ]:
import os, sys, json, time, pickle, hashlib, platform
from importlib.metadata import version as package_version
from pathlib import Path

PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform

from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, sample_lg_point, sample_lg_marginalized,
    calibration_summary, require_legacy_environment)
from splits import group_split

require_legacy_environment(PROJECT)
RESULTS_DIR = configure_run(PROJECT)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)
SEEDS = tuple(int(s.strip()) for s in SEEDS_CSV.split(",") if s.strip())
if not SEEDS:
    raise ValueError("SEEDS_CSV must contain at least one integer seed.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project: {PROJECT}")
print(f"Results: {RESULTS_DIR}")
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Seeds: {SEEDS}")

## Reconstruct the exact FLAMINGO training population

The primary models were trained after complete-case filtering on the full feature collection. The no-ratio model deliberately retains that same parent population rather than admitting extra rows, so the feature removal is the only data change.


In [ ]:
CACHE_DIR = PROJECT / "analogues/cache/run_001_mstar2e10"
with open(CACHE_DIR / "flamingo_sample.pkl", "rb") as f:
    sample = pickle.load(f)
with open(CACHE_DIR / "flamingo_pairs.pkl", "rb") as f:
    pairs = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

pi, pj = pairs["i"], pairs["j"]
gr_i, gr_j = sample["g_r"][pi], sample["g_r"][pj]
sigma_i, sigma_j = sample["vel_disp"][pi], sample["vel_disp"][pj]
kappa_i, kappa_j = sample["kappa_rot"][pi], sample["kappa_rot"][pj]
met_i = met_transform["flamingo_met_dex"][pi]
met_j = met_transform["flamingo_met_dex"][pj]

flam_df = pd.DataFrame({
    "separation": pairs["separation_kpc"],
    "vr": pairs["vr"],
    "vt": pairs["vt"],
    "log_mass_ratio": pairs["log_mass_ratio"],
    "sigma_mean": (sigma_i + sigma_j) / 2,
    "sigma_diff": np.abs(sigma_i - sigma_j),
    "kappa_mean": (kappa_i + kappa_j) / 2,
    "kappa_diff": np.abs(kappa_i - kappa_j),
    "met_diff": np.abs(met_i - met_j),
    "gr_mean": (gr_i + gr_j) / 2,
    "gr_diff": np.abs(gr_i - gr_j),
    "log_m200c": pairs["log_tot_virial_mass"],
    "host_i": sample["host_top_index"][pi],
    "host_j": sample["host_top_index"][pj],
    "same_host": pairs["have_same_host"].astype(bool),
})

ALL_FEATURES = ["separation", "vr", "vt", "log_mass_ratio",
                "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                "met_diff", "gr_mean", "gr_diff"]
TARGET = "log_m200c"
flam_df_clean = flam_df.dropna(subset=ALL_FEATURES + [TARGET])
flam_diff = flam_df_clean.loc[~flam_df_clean["same_host"]].reset_index(drop=True)

if len(flam_df_clean) != EXPECTED_FLAMINGO_FULL or len(flam_diff) != EXPECTED_FLAMINGO_DIFF_HOST:
    raise RuntimeError(
        "FLAMINGO population differs from the frozen headline population: "
        f"full={len(flam_df_clean):,}, diff_host={len(flam_diff):,}."
    )

train_idx, test_idx = group_split(flam_diff, test_size=0.15, seed=42)
split_digest = hashlib.sha256()
split_digest.update(np.asarray(train_idx, dtype=np.int64).tobytes())
split_digest.update(np.asarray(test_idx, dtype=np.int64).tobytes())
SPLIT_SHA256 = split_digest.hexdigest()

print(f"FLAMINGO complete cases: {len(flam_df_clean):,}")
print(f"Different-host cases: {len(flam_diff):,}")
print(f"Train/test: {len(train_idx):,}/{len(test_idx):,}")
print(f"Split SHA256: {SPLIT_SHA256}")

In [ ]:
FEATURE_SETS = {
    "kinematics_only": ["separation", "vr", "vt"],
    "ratio_conditioned": ["separation", "vr", "vt", "log_mass_ratio"],
}

with open(LEGACY_RESULTS_DIR / "optuna_search_results.pkl", "rb") as f:
    best_params = pickle.load(f)["best_params"]

MODEL_SPECS = {
    "maf": {
        "model": "maf", "hidden_features": best_params["hidden_features"],
        "num_transforms": best_params["num_transforms"],
    },
    "nsf": {
        "model": "nsf", "hidden_features": best_params["hidden_features"],
        "num_transforms": best_params["num_transforms"], "num_bins": 10,
    },
    "mdn": {
        "model": "mdn", "hidden_features": best_params["hidden_features"],
        "num_components": 10,
    },
}
TRAINING_SPEC = {
    "batch_size": best_params["batch_size"],
    "learning_rate": best_params["learning_rate"],
    "max_num_epochs": MAX_NUM_EPOCHS,
    "stop_after_epochs": STOP_AFTER_EPOCHS,
}

prior = BoxUniform(
    low=torch.tensor([11.5], device=device),
    high=torch.tensor([14.5], device=device),
)
print("Best training parameters:", best_params)
print("Model specifications:", json.dumps(MODEL_SPECS, indent=2))

In [ ]:
def atomic_pickle_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
    tmp.replace(path)


def train_one_model(train_df, feats, model_spec, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_raw = train_df[feats].values.astype(np.float32)
    y_raw = train_df[TARGET].values.astype(np.float32)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_raw)
    density_builder = posterior_nn(**model_spec)
    inference = SNPE(prior=prior, density_estimator=density_builder, device=device)
    inference.append_simulations(
        torch.tensor(y_raw.reshape(-1, 1), device=device),
        torch.tensor(X_sc, device=device),
    )
    density_estimator = inference.train(
        training_batch_size=TRAINING_SPEC["batch_size"],
        learning_rate=TRAINING_SPEC["learning_rate"],
        max_num_epochs=TRAINING_SPEC["max_num_epochs"],
        stop_after_epochs=TRAINING_SPEC["stop_after_epochs"],
        show_train_summary=False,
    )
    return inference.build_posterior(density_estimator), density_estimator, scaler


def posterior_draws_in_condition_batches(posterior, X_sc, n_draws, seed, batch_size):
    torch.manual_seed(seed)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(X_sc), batch_size):
            x = torch.tensor(X_sc[start:start + batch_size], device=device)
            draws = posterior.sample_batched((n_draws,), x=x).squeeze(-1)
            chunks.append(draws.cpu().numpy())
    return np.concatenate(chunks, axis=1)


def summarize_samples(samples):
    q16, med, q84 = np.percentile(samples, [16, 50, 84])
    return {"median": float(med), "p16": float(q16), "p84": float(q84),
            "width68": float(q84 - q16), "std": float(np.std(samples))}

## Train, evaluate, and checkpoint

The checkpoint is written atomically after every fit. If a Slurm allocation ends, submit the notebook again with the same parameters; completed keys will be skipped.


In [ ]:
CHECKPOINT_PATH = RESULTS_DIR / "npe_model_family_checkpoint.pkl"
PREDICTIVE_CSV = RESULTS_DIR / "npe_model_family_predictive_metrics.csv"
LG_CSV = RESULTS_DIR / "npe_model_family_lg_summary.csv"
COMPARISON_CSV = RESULTS_DIR / "npe_model_family_feature_contrast.csv"
SUMMARY_PATH = RESULTS_DIR / "npe_model_family_summary.pkl"
FIGURE_PATH = RESULTS_DIR / "npe_model_family_posteriors.pdf"

signature_payload = {
    "run_version": RUN_VERSION, "features": FEATURE_SETS, "models": MODEL_SPECS,
    "training": TRAINING_SPEC, "seeds": SEEDS, "split_sha256": SPLIT_SHA256,
    "n_posterior_draws": N_POSTERIOR_DRAWS, "n_lg_samples": N_LG_SAMPLES,
    "n_obs_draws": N_OBS_DRAWS, "n_obs_samples_per_draw": N_OBS_SAMPLES_PER_DRAW,
}
EXPERIMENT_SHA256 = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()

state = {
    "posteriors": {}, "scalers": {}, "predictive_metrics": {},
    "lg_samples": {}, "lg_summary": {},
    "metadata": {
        **signature_payload, "experiment_sha256": EXPERIMENT_SHA256,
        "model_description": MODEL_DESCRIPTION, "population": "flamingo_diff_host",
        "n_full": len(flam_df_clean), "n_diff_host": len(flam_diff),
        "n_train": len(train_idx), "n_test": len(test_idx),
        "python": platform.python_version(), "torch": torch.__version__,
        "sbi": package_version("sbi"), "sklearn": package_version("scikit-learn"),
    },
}
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as f:
        saved = pickle.load(f)
    saved_signature = saved.get("metadata", {}).get("experiment_sha256")
    if saved_signature != EXPERIMENT_SHA256:
        raise RuntimeError(
            f"Existing checkpoint has incompatible signature {saved_signature}; "
            f"expected {EXPERIMENT_SHA256}. Move it aside or restore the original parameters."
        )
    state = saved
    print(f"Resuming {CHECKPOINT_PATH}: {len(state['posteriors'])} fits completed.")
else:
    print(f"Starting new experiment: {EXPERIMENT_SHA256}")

In [ ]:
train_df = flam_diff.iloc[train_idx]
test_df = flam_diff.iloc[test_idx]
run_times = []
n_total = len(SEEDS) * len(FEATURE_SETS) * len(MODEL_SPECS)

for seed in SEEDS:
    for feature_name, feats in FEATURE_SETS.items():
        for model_name, model_spec in MODEL_SPECS.items():
            key = f"{feature_name}__{model_name}__seed{seed}"
            if key in state["posteriors"]:
                print(f"Skipping completed fit: {key}")
                continue

            t0 = time.time()
            print(f"\nTraining {key} ({len(state['posteriors']) + 1}/{n_total})")
            posterior, density_estimator, scaler = train_one_model(
                train_df, feats, model_spec, seed
            )

            X_test_sc = scaler.transform(test_df[feats].values.astype(np.float32))
            y_test = test_df[TARGET].values.astype(np.float32)
            with torch.no_grad():
                test_log_prob = density_estimator.log_prob(
                    torch.tensor(y_test.reshape(-1, 1), device=device),
                    condition=torch.tensor(X_test_sc, device=device),
                ).mean().item()

            test_draws = posterior_draws_in_condition_batches(
                posterior, X_test_sc, N_POSTERIOR_DRAWS,
                seed=50_000 + seed, batch_size=CALIBRATION_BATCH_SIZE,
            )
            predictions = test_draws.mean(axis=0)
            cal = calibration_summary(test_draws, y_test)
            predictive = {
                "key": key, "feature_set": feature_name, "model": model_name,
                "seed": seed, "n_features": len(feats), "n_train": len(train_df),
                "n_test": len(test_df), "test_log_prob": float(test_log_prob),
                "r2": float(r2_score(y_test, predictions)),
                "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
                "bias": float(np.mean(predictions - y_test)),
                "coverage68": cal["coverage68"], "coverage95": cal["coverage95"],
                "rank_cdf_distance": cal["rank_cdf_distance"],
                "mean_width68": float(cal["mean_interval_width"][1]),
                "median_beta": cal["median_beta"],
            }

            model_lg_samples = {}
            model_lg_summary = {}
            for scenario_idx, scenario in enumerate(("vdM2012", "Salomon2021")):
                obs, unc = lg_observation(scenario)
                torch.manual_seed(60_000 + 100 * seed + scenario_idx)
                point = sample_lg_point(
                    posterior, scaler, feats, obs, h0=H0_BY_SIM["flamingo"],
                    device=device, n=N_LG_SAMPLES,
                )
                point_key = f"{scenario}__point"
                model_lg_samples[point_key] = point
                model_lg_summary[point_key] = summarize_samples(point)

                if scenario == "Salomon2021":
                    torch.manual_seed(70_000 + seed)
                    marginalized = sample_lg_marginalized(
                        posterior, scaler, feats, obs, unc,
                        h0=H0_BY_SIM["flamingo"], device=device,
                        n_draws=N_OBS_DRAWS, n_per=N_OBS_SAMPLES_PER_DRAW, seed=0,
                    )
                    marg_key = f"{scenario}__marginalized"
                    model_lg_samples[marg_key] = marginalized
                    model_lg_summary[marg_key] = summarize_samples(marginalized)

            elapsed = time.time() - t0
            predictive["elapsed_minutes"] = elapsed / 60
            state["posteriors"][key] = posterior
            state["scalers"][key] = scaler
            state["predictive_metrics"][key] = predictive
            state["lg_samples"][key] = model_lg_samples
            state["lg_summary"][key] = model_lg_summary
            atomic_pickle_dump(state, CHECKPOINT_PATH)

            run_times.append(elapsed)
            sal = model_lg_summary["Salomon2021__marginalized"]
            remaining = n_total - len(state["posteriors"])
            eta = np.mean(run_times) * remaining / 3600 if run_times else np.nan
            print(
                f"  logp={test_log_prob:.4f}, R2={predictive['r2']:.4f}, "
                f"coverage68={predictive['coverage68']:.3f}"
            )
            print(
                f"  Salomon marginalized: {sal['median']:.3f} "
                f"[{sal['p16']:.3f}, {sal['p84']:.3f}]"
            )
            print(f"  took {elapsed/60:.1f} min; checkpointed; estimated remaining {eta:.1f} h")

print(f"\nCompleted {len(state['posteriors'])}/{n_total} fits.")

## Summaries and direct feature contrast

The contrast is calculated within each estimator family and seed. A negative median delta means adding the mass ratio lowers the inferred log mass; a width ratio below one means it narrows the posterior.


In [ ]:
predictive_df = pd.DataFrame(state["predictive_metrics"].values()).sort_values(
    ["feature_set", "model", "seed"]
)

lg_rows = []
for key, contexts in state["lg_summary"].items():
    pred = state["predictive_metrics"][key]
    for context, stats in contexts.items():
        scenario, conditioning = context.split("__", 1)
        lg_rows.append({
            "key": key, "feature_set": pred["feature_set"],
            "model": pred["model"], "seed": pred["seed"],
            "scenario": scenario, "conditioning": conditioning, **stats,
        })
lg_df = pd.DataFrame(lg_rows).sort_values(
    ["scenario", "conditioning", "model", "seed", "feature_set"]
)

contrast_index = ["model", "seed", "scenario", "conditioning"]
medians = lg_df.pivot(index=contrast_index, columns="feature_set", values="median")
widths = lg_df.pivot(index=contrast_index, columns="feature_set", values="width68")
contrast_df = medians.add_suffix("_median").join(widths.add_suffix("_width68")).reset_index()
contrast_df["delta_median_ratio_minus_kinematics"] = (
    contrast_df["ratio_conditioned_median"] - contrast_df["kinematics_only_median"]
)
contrast_df["width_ratio_conditioned_over_kinematics"] = (
    contrast_df["ratio_conditioned_width68"] / contrast_df["kinematics_only_width68"]
)

predictive_df.to_csv(PREDICTIVE_CSV, index=False)
lg_df.to_csv(LG_CSV, index=False)
contrast_df.to_csv(COMPARISON_CSV, index=False)
atomic_pickle_dump({
    "predictive_metrics": predictive_df, "lg_summary": lg_df,
    "feature_contrast": contrast_df, "metadata": state["metadata"],
}, SUMMARY_PATH)

print("\nPredictive metrics:")
display(predictive_df[["feature_set", "model", "seed", "test_log_prob",
                       "r2", "rmse", "bias", "coverage68",
                       "coverage95", "mean_width68"]])
print("\nSalomon2021 marginalized Local Group results:")
display(lg_df.query("scenario == 'Salomon2021' and conditioning == 'marginalized'")[[
    "feature_set", "model", "seed", "median", "p16", "p84", "width68"
]])
print("\nWithin-family effect of adding log_mass_ratio:")
display(contrast_df)

In [ ]:
# Load the current saved headline artifact for an explicit side-by-side reference.
headline_path = RESULTS_DIR / "uncertainty_marginalized_results.pkl"
if headline_path.exists():
    with open(headline_path, "rb") as f:
        headline_artifact = pickle.load(f)
    headline_key = "core_dynamics__diff_host"
    if headline_key in headline_artifact:
        existing_headline = pd.DataFrame([
            {"result": "existing ratio-conditioned MAF, Salomon point",
             **summarize_samples(headline_artifact[headline_key]["point"])},
            {"result": "existing ratio-conditioned MAF, Salomon marginalized",
             **summarize_samples(headline_artifact[headline_key]["marginalized"])},
        ])
        display(existing_headline)
    else:
        print(f"{headline_key} is absent from {headline_path}")
else:
    print(f"Existing headline artifact not found: {headline_path}")

In [ ]:
plot_contexts = [("vdM2012__point", "vdM2012 point"),
                 ("Salomon2021__marginalized", "Salomon2021 marginalized")]
colors = {"kinematics_only": "tab:blue", "ratio_conditioned": "tab:orange"}
fig, axes = plt.subplots(len(plot_contexts), len(MODEL_SPECS),
                         figsize=(15, 7), sharex=True, sharey="row")
xs = np.linspace(11.5, 14.5, 500)
for row, (context, context_label) in enumerate(plot_contexts):
    for col, model_name in enumerate(MODEL_SPECS):
        ax = axes[row, col]
        for feature_name in FEATURE_SETS:
            for seed_idx, seed in enumerate(SEEDS):
                key = f"{feature_name}__{model_name}__seed{seed}"
                if key not in state["lg_samples"]:
                    continue
                values = state["lg_samples"][key][context]
                kde = gaussian_kde(values, bw_method=0.15)
                ax.plot(xs, kde(xs), color=colors[feature_name],
                        alpha=0.85 if seed_idx == 0 else 0.25,
                        lw=2 if seed_idx == 0 else 1,
                        label=feature_name if seed_idx == 0 else None)
        ax.set_title(f"{model_name.upper()} — {context_label}")
        if col == 0:
            ax.set_ylabel("density")
        if row == len(plot_contexts) - 1:
            ax.set_xlabel(r"$\log_{10}(M_{200c}/M_\odot)$")
        ax.legend(fontsize=8)
fig.suptitle("Different-host Local Group posterior: model family and mass-ratio conditioning")
fig.tight_layout()
fig.savefig(FIGURE_PATH, bbox_inches="tight")
plt.show()
print(f"Saved figure: {FIGURE_PATH}")

## Reporting guardrails

1. Compare kinematics-only and ratio-conditioned posteriors within the same model family and seed.
2. Treat seed spread as training variability; do not select the seed or estimator giving the preferred Local Group mass.
3. Prefer held-out log probability and empirical coverage when judging density estimators. R-squared alone does not assess posterior shape.
4. Keep the existing MAF headline unless a flexible family improves held-out density estimation and calibration consistently across seeds.
5. If NSF or MDN changes the scientific conclusion materially, only then propagate the selected family into the broader SBC, prior-sensitivity, and bootstrap notebooks.
